# Jobs: beta + K

project = ```BayLearn```, host = ```any```, device = ```any```

**Motivation**: <br>


In [1]:
# HIDE CODE


import os, sys
from IPython.display import display

# tmp & extras dir
git_dir = os.path.join(os.environ['HOME'], 'Dropbox/git')
extras_dir = os.path.join(git_dir, 'jb-progress-2025/_extras')
fig_base_dir = os.path.join(git_dir, 'jb-progress-2025/figs')
tmp_dir = os.path.join(git_dir, 'jb-progress-2025/tmp')

# GitHub
sys.path.insert(0, os.path.join(git_dir, '_TemporalSC'))
from figures.convergence import plot_convergence
from main.config_defaults import default_configs
from figures.fighelper import *
from main.train import *

# warnings, tqdm, & style
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
from rich.jupyter import print
%matplotlib inline
set_style()

## Setup

In [2]:
from base.helper import job_runner_script


def _cleanup(path, host=None):
    for f in os.listdir(path):
        cond = f.endswith('.txt')
        if host is not None:
            cond = cond and host in f
        if cond:
            os.remove(pjoin(path, f))


def _name(host, gpu_i, fit_i):
    return f"{host}-cuda{gpu_i}-fit{fit_i}"

In [3]:
save_dir = 'Dropbox/git/_TemporalSC/scripts'
save_dir = pjoin(os.environ['HOME'], save_dir)
os.makedirs(save_dir, exist_ok=True)

# delete existing job runners?
_cleanup(save_dir, None)

print(sorted(os.listdir(save_dir)))

[
    'cleanup_chkpts.sh',
    'cleanup_recursive.sh',
    'copyfits.sh',
    'fit_model.sh',
    'kill_screens.sh',
    'resume_fit.sh',
    'run_sessions.sh',
    'test_tqdm.py',
    'test_tqdm.sh'
]

## chewie

In [4]:
host = 'chewie'
_cleanup(save_dir, host)

scripts = collections.defaultdict(list)
tot = 0

In [5]:
model_type = 'poisson'
t_train = 16

dataset = 'Kyoto16-wht'  # 'vH16-wht'
latent_dims = [32, 64, 128, 512, 768, 1024, 2048, 4096]
betas = [
    0.1, 1, 2, 4, 8,
    # 12, 16, 20, 24, 32,
]

len(betas), len(latent_dims)

(5, 8)

In [6]:
for k in latent_dims:
    for kl_beta in betas:
        # get arg
        arg = ' '.join([
            f"--t_train {t_train}",
            f"--kl_beta {kl_beta}",
            f"--latent_channels {k}",
            '--comment BayLearn',
            '--verbose',
            # '--dry_run',
        ])
        gpu_i = tot % torch.cuda.device_count()
        scripts[gpu_i].append(job_runner_script(
            device=gpu_i,
            dataset=dataset,
            model=model_type,
            archi='ngd|lin',
            args=arg,
            seed=0,
        ))
        tot += 1

In [7]:
print(tot)

40

In [8]:
scripts = dict(scripts)
print({k: len(v) for k, v in scripts.items()})

{0: 20, 1: 20}

### Save

In [9]:
n_fits = 5

for gpu_i, scripts_list in scripts.items():
    scripts_divided = divide_list(scripts_list, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
        )
        print(combined.replace('&& ', '&& \n'))

[PROGRESS] 'chewie-cuda0-fit0.txt' saved at
/home/hadi/Dropbox/git/_TemporalSC/scripts

./fit_model.sh '0' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 0.1 --latent_channels 32 
--comment BayLearn --verbose && 
./fit_model.sh '0' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 2 --latent_channels 32 
--comment BayLearn --verbose && 
./fit_model.sh '0' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 8 --latent_channels 32 
--comment BayLearn --verbose && 
./fit_model.sh '0' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 1 --latent_channels 64 
--comment BayLearn --verbose

[PROGRESS] 'chewie-cuda0-fit1.txt' saved at
/home/hadi/Dropbox/git/_TemporalSC/scripts

./fit_model.sh '0' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 4 --latent_channels 64 
--comment BayLearn --verbose && 
./fit_model.sh '0' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 0.1 --latent_channels 128 
--comment BayLearn --verbose && 
./fit_model.sh '0' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 2 --latent_channels 128 
--comment BayLearn --verbose && 
./fit_model.sh '0' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 8 --latent_channels 128 
--comment BayLearn --verbose

[PROGRESS] 'chewie-cuda0-fit2.txt' saved at
/home/hadi/Dropbox/git/_TemporalSC/scripts

...

./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 2 --latent_channels 512 
--comment BayLearn --verbose && 
./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 8 --latent_channels 512 
--comment BayLearn --verbose && 
./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 1 --latent_channels 768 
--comment BayLearn --verbose && 
./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 4 --latent_channels 768 
--comment BayLearn --verbose

[PROGRESS] 'chewie-cuda1-fit3.txt' saved at
/home/hadi/Dropbox/git/_TemporalSC/scripts

./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 0.1 --latent_channels 1024 
--comment BayLearn --verbose && 
./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 2 --latent_channels 1024 
--comment BayLearn --verbose && 
./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 8 --latent_channels 1024 
--comment BayLearn --verbose && 
./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 1 --latent_channels 2048 
--comment BayLearn --verbose

[PROGRESS] 'chewie-cuda1-fit4.txt' saved at
/home/hadi/Dropbox/git/_TemporalSC/scripts

./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 4 --latent_channels 2048 
--comment BayLearn --verbose && 
./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 0.1 --latent_channels 4096 
--comment BayLearn --verbose && 
./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 2 --latent_channels 4096 
--comment BayLearn --verbose && 
./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 8 --latent_channels 4096 
--comment BayLearn --verbose

Print one to check

In [10]:
print(combined.replace('&& ', '&& \n'))

./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 4 --latent_channels 2048 
--comment BayLearn --verbose && 
./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 0.1 --latent_channels 4096 
--comment BayLearn --verbose && 
./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 2 --latent_channels 4096 
--comment BayLearn --verbose && 
./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 8 --latent_channels 4096 
--comment BayLearn --verbose

## yoru

In [4]:
host = 'yoru'
_cleanup(save_dir, host)

scripts = collections.defaultdict(list)
tot = 0

In [5]:
model_type = 'poisson'
t_train = 16

dataset = 'Kyoto16-wht'  # 'vH16-wht'
latent_dims = [32, 64, 128, 512, 768, 1024, 2048, 4096]
betas = [
    # 0.1, 1, 2, 4, 8,
    12, 16, 20, 24, 32,
]

len(betas), len(latent_dims)

(5, 8)

In [6]:
for k in latent_dims:
    for kl_beta in betas:
        # get arg
        arg = ' '.join([
            f"--t_train {t_train}",
            f"--kl_beta {kl_beta}",
            f"--latent_channels {k}",
            '--comment BayLearn',
            '--verbose',
            # '--dry_run',
        ])
        gpu_i = tot % torch.cuda.device_count()
        scripts[gpu_i].append(job_runner_script(
            device=gpu_i,
            dataset=dataset,
            model=model_type,
            archi='ngd|lin',
            args=arg,
            seed=0,
        ))
        tot += 1

In [7]:
print(tot)

40

In [8]:
scripts = dict(scripts)
print({k: len(v) for k, v in scripts.items()})

{0: 20, 1: 20}

### Save

In [9]:
n_fits = 5

for gpu_i, scripts_list in scripts.items():
    scripts_divided = divide_list(scripts_list, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
        )
        print(combined.replace('&& ', '&& \n'))

[PROGRESS] 'yoru-cuda0-fit0.txt' saved at
/home/hadi/Dropbox/git/_TemporalSC/scripts

./fit_model.sh '0' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 12 --latent_channels 32 
--comment BayLearn --verbose && 
./fit_model.sh '0' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 20 --latent_channels 32 
--comment BayLearn --verbose && 
./fit_model.sh '0' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 32 --latent_channels 32 
--comment BayLearn --verbose && 
./fit_model.sh '0' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 16 --latent_channels 64 
--comment BayLearn --verbose

[PROGRESS] 'yoru-cuda0-fit1.txt' saved at
/home/hadi/Dropbox/git/_TemporalSC/scripts

./fit_model.sh '0' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 24 --latent_channels 64 
--comment BayLearn --verbose && 
./fit_model.sh '0' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 12 --latent_channels 128 
--comment BayLearn --verbose && 
./fit_model.sh '0' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 20 --latent_channels 128 
--comment BayLearn --verbose && 
./fit_model.sh '0' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 32 --latent_channels 128 
--comment BayLearn --verbose

[PROGRESS] 'yoru-cuda0-fit2.txt' saved at
/home/hadi/Dropbox/git/_TemporalSC/scripts

...

./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 20 --latent_channels 512 
--comment BayLearn --verbose && 
./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 32 --latent_channels 512 
--comment BayLearn --verbose && 
./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 16 --latent_channels 768 
--comment BayLearn --verbose && 
./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 24 --latent_channels 768 
--comment BayLearn --verbose

[PROGRESS] 'yoru-cuda1-fit3.txt' saved at
/home/hadi/Dropbox/git/_TemporalSC/scripts

./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 12 --latent_channels 1024 
--comment BayLearn --verbose && 
./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 20 --latent_channels 1024 
--comment BayLearn --verbose && 
./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 32 --latent_channels 1024 
--comment BayLearn --verbose && 
./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 16 --latent_channels 2048 
--comment BayLearn --verbose

[PROGRESS] 'yoru-cuda1-fit4.txt' saved at
/home/hadi/Dropbox/git/_TemporalSC/scripts

./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 24 --latent_channels 2048 
--comment BayLearn --verbose && 
./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 12 --latent_channels 4096 
--comment BayLearn --verbose && 
./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 20 --latent_channels 4096 
--comment BayLearn --verbose && 
./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 32 --latent_channels 4096 
--comment BayLearn --verbose

Print one to check

In [10]:
print(combined.replace('&& ', '&& \n'))

./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 24 --latent_channels 2048 
--comment BayLearn --verbose && 
./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 12 --latent_channels 4096 
--comment BayLearn --verbose && 
./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 20 --latent_channels 4096 
--comment BayLearn --verbose && 
./fit_model.sh '1' 'Kyoto16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 32 --latent_channels 4096 
--comment BayLearn --verbose